In [ ]:
import sys
!{sys.executable} -m pip install seaborn

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('ATP_W152_clean.csv')
df.head()

## Step 1: Define Feature Matrices

We use a hybrid encoding strategy:
- **Ordinal predictors** (`educ_6cat`, `inc_9cat`, `age_cat`) are kept as raw integers — their numeric ordering is meaningful, so dummy coding would discard that information.
- **Nominal predictors** (`gender`, `race`) are dummy-coded — there is no meaningful numeric ordering to these categories, so treating them as integers would imply a false ranking (e.g., race=3 is not "more" of anything than race=1).

Encoding is defined here, before any modeling, so all subsequent models use correctly-specified inputs.

In [ ]:
# Socioeconomic-only feature matrix (ordinal, no dummies needed)
X_socio = df[['educ_6cat', 'inc_9cat']]

# Dummy-encode nominal variables only
demos_encoded = pd.get_dummies(
    df[['gender', 'race']],
    columns=['gender', 'race'],
    drop_first=True  # drop one level per variable to avoid perfect multicollinearity
)

# Full feature matrix: ordinal predictors as-is + dummies for nominal variables
X_full = pd.concat([
    df[['educ_6cat', 'inc_9cat', 'age_cat']],
    demos_encoded
], axis=1)

# Outcomes
y_binary = df['chatuse_binary']
y_freq   = df['useai']

print("X_socio shape:", X_socio.shape)
print("X_full shape: ", X_full.shape)
print("\nX_full columns:", list(X_full.columns))

## Step 2: Baseline

Before modeling, establish the naive classifier baseline — the accuracy a model would achieve by always predicting the majority class.

In [ ]:
# Naive baseline: majority-class accuracy
print("Class proportions (chatuse_binary):")
print(y_binary.value_counts(normalize=True))
print("\nBaseline accuracy (always predict majority class):", round(y_binary.value_counts(normalize=True).max(), 4))

## Step 3: Ordinal vs. Dummy Encoding Comparison (Socioeconomic Model)

We replicate the original comparison to confirm that ordinal encoding and dummy encoding produce equivalent accuracy for Random Forest on these ordinal predictors. This motivates using ordinal encoding for `educ_6cat` and `inc_9cat` throughout.

In [ ]:
# Random Forest: ordinal encoding (no dummies)
X_train, X_test, y_train, y_test = train_test_split(X_socio, y_binary, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc_ordinal = accuracy_score(y_test, rf.predict(X_test))
print("Accuracy (ordinal encoding):", round(acc_ordinal, 4))

# Random Forest: dummy encoding for comparison
X_dummy = pd.get_dummies(df[['educ_6cat', 'inc_9cat']], drop_first=True)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_dummy, y_binary, test_size=0.2, random_state=42)
rf_d = RandomForestClassifier(n_estimators=100, random_state=42)
rf_d.fit(X_train_d, y_train_d)
acc_dummy = accuracy_score(y_test_d, rf_d.predict(X_test_d))
print("Accuracy (dummy encoding): ", round(acc_dummy, 4))

print("\nConclusion: Ordinal encoding is equivalent and preferred — no dummy variables needed for ordinal predictors.")

## Step 4: Cross-Validated Model Comparison

A single train/test split is sensitive to which observations happen to land in the test set. 5-fold cross-validation gives a more stable estimate of generalization accuracy and lets us compare models with a mean ± SD rather than a single number.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Socioeconomic model (education + income only)
cv_socio = cross_val_score(rf, X_socio, y_binary, cv=5, scoring='accuracy')

# Full model (education + income + age + gender + race, correctly encoded)
cv_full = cross_val_score(rf, X_full, y_binary, cv=5, scoring='accuracy')

print("5-Fold Cross-Validation Accuracy")
print("-" * 45)
print(f"Socioeconomic only  — Mean: {cv_socio.mean():.3f}, SD: {cv_socio.std():.3f}")
print(f"Full model          — Mean: {cv_full.mean():.3f}, SD: {cv_full.std():.3f}")
print(f"Naive baseline      — Mean: {y_binary.value_counts(normalize=True).max():.3f}")

## Step 5: Adoption Probability Heatmap

We fit the socioeconomic model on the full dataset (not a train/test split) and use `predict_proba` to visualize the predicted probability of AI chatbot adoption across every combination of income and education level. This directly illustrates the privilege gradient hypothesized in the proposal.

In [ ]:
# Fit on full data — this is for visualization, not evaluation
rf_viz = RandomForestClassifier(n_estimators=100, random_state=42)
rf_viz.fit(X_socio, y_binary)

# Build the prediction grid
educ_vals = sorted(df['educ_6cat'].unique())  # 1–6
inc_vals  = sorted(df['inc_9cat'].unique())   # 1–9

grid = pd.DataFrame(
    [(e, i) for e in educ_vals for i in inc_vals],
    columns=['educ_6cat', 'inc_9cat']
)
grid['prob_adopt'] = rf_viz.predict_proba(grid)[:, 1]

heatmap_data = grid.pivot(index='educ_6cat', columns='inc_9cat', values='prob_adopt')

# Readable axis labels
educ_labels = ['< HS', 'HS grad', 'Some college', 'Associate', 'College grad', 'Postgrad']
inc_labels  = ['<$30k', '$30-40k', '$40-50k', '$50-60k', '$60-70k', '$70-80k', '$80-100k', '$100-130k', '$130k+']

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    vmin=0, vmax=1,
    xticklabels=inc_labels,
    yticklabels=educ_labels,
    ax=ax
)
ax.set_title('Predicted Probability of AI Chatbot Adoption\nby Education and Income Level', fontsize=13)
ax.set_xlabel('Family Income')
ax.set_ylabel('Education Level')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 6: Frequency of Use — Spearman Correlation

`useai` is a 5-point ordinal scale, so MSE and R² from a regressor are not meaningful metrics (they assume equal-interval distances between response categories). Instead, we use Spearman rank correlation to assess whether income and education are monotonically associated with AI use frequency — a more appropriate test for ordinal outcomes.

In [ ]:
from scipy.stats import spearmanr

rho_educ, p_educ = spearmanr(df['educ_6cat'], df['useai'])
rho_inc,  p_inc  = spearmanr(df['inc_9cat'],  df['useai'])

print("Spearman Correlations with AI Use Frequency (useai)")
print("-" * 50)
print(f"Education (educ_6cat): rho = {rho_educ:.3f}, p = {p_educ:.4f}")
print(f"Income    (inc_9cat):  rho = {rho_inc:.3f},  p = {p_inc:.4f}")
print("\nNote: useai is a 5-point ordinal scale (1=Almost constantly, 5=Less often).")
print("Negative rho means higher SES is associated with more frequent use (lower score).")

## Step 7: Feature Importance

Random Forest feature importances show the relative contribution of each predictor to reducing classification error across all trees. This directly addresses Hypothesis 1: whether education or income is the stronger predictor of adoption.

In [ ]:
# Fit full model on all data for importance extraction
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_full, y_binary)

importances = pd.Series(rf_full.feature_importances_, index=X_full.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances — Full Random Forest Model')
ax.set_xlabel('Mean Decrease in Impurity')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nImportance values:")
print(importances.sort_values(ascending=False).round(3))